# N019 · 哈希查找与索引映射

**目标：** 将两重枚举降为一次扫描和补数查询。

**先修：** N007, N015, N017。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 值到索引；先查后存；重复元素；平均复杂度假设。

**配套讲解来源：** [同名逐章意见](../../comment/019_hash_lookup_index.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章的主角是那道人人都刷过的题：**两数之和**。给定一个整数数组 `nums` 和目标值 `target`，你要找出两个下标不同的位置 i 和 j，使得 `nums[i]+nums[j]` 恰好等于 target，返回这对下标。注意约束：同一个元素不许用两次（也就是 i 和 j 必须是两个不同的位置）；如果有多组解，返回任意一组都算对；如果无解，本章约定返回 `None`。

举个具体到数字的例子：输入 `nums=[2,7,11,15]`、`target=9` → 输出 `[0,1]` → 为什么？因为 `nums[0]+nums[1]=2+7=9`，恰好等于目标。

再看一个专门考察细节的例子：输入 `nums=[3,3]`、`target=6` → 输出 `[0,1]` → 为什么？两个 3 虽然值相同，但它们是两个不同位置的元素，`3+3=6` 用的是下标 0 的 3 和下标 1 的 3，合法。这个例子逼你的实现必须"按位置区分元素"，而不能只按值思考。

先把两条路线的差距摆出来：

| 路线 | 核心动作 | 时间 | 空间 |
|---|---|---|---|
| 暴力两重循环 | 对每个 i，把右边所有 j 挨个试一遍 | O(n²) | O(1) |
| 哈希一遍扫描 | 对每个 x，用字典一步查补数在不在 | 平均 O(n) | O(n) |

n=10⁴ 时两者相差约 2500 倍，哈希路线多花的只是一个存下标的字典。

## 2. 基础知识：先读懂这些词

- **两数之和问题**：上面描述的"找两个不同位置的元素凑出目标和"的问题，对应 LeetCode 1 题。
- **哈希表 / 字典（dict）**：一种"键→值"的极速查找结构。给定键，平均只需常数时间就能查到（或查不到）对应的值——这正是它比"在列表里一个个找"快的原因。
- **值到索引映射（value→index map）**：本章字典的用法——键存"元素的值"，值存"这个值出现过的下标"。这样拿到 target−x 之后，一查字典就知道"这个数出现过没有、在哪"。
- **补数（complement）**：对于当前元素 x，target−x 就是它的"补数"——x 需要的那个搭档。例如 target=9、当前 x=7，补数就是 2。
- **先查后存（query before insert）**：在每一轮里，先用当前 x 查字典，再把 x 存进字典。顺序颠倒会导致"x 自己和自己配对"的错误（细节见第三节手算）。
- **只查询过去**：字典里永远只存"当前元素之前"出现过的元素。这个约定从结构上保证了配对双方下标一定不同——搭档一定在左边，当前元素在右边。
- **平均复杂度假设**：哈希查找"常数时间"是平均意义下的。绝大多数真实输入都符合这个假设，但理论上存在哈希冲突严重的极端输入。我们说哈希版是"平均 O(n)"，说的就是这个前提。

## 3. 思路推导：从小例子开始

- **Step 1：先想暴力怎么做。** 两层循环枚举所有下标对 (i,j)（i<j），逐一检查和是否等于 target。它正确但慢：n 个元素要对约 n²/2 对做检查。
- **Step 2：找出浪费在哪。** 暴力的内层循环其实在回答"数组前面有没有 target−x 这个值"。这是一个查找问题，而查找恰恰是哈希表的看家本领——查一次只要常数时间。
- **Step 3：边扫边记账。** 扫描数组，每到一个新位置 j，先问字典："target−nums[j] 在你那里吗？"在，就当场配对成功返回；不在，就把"值 nums[j] → 下标 j"记进字典，继续往右走。
- **Step 4：先查后存。** 每轮必须先查询、再把自己存进去。这样当前元素查询时字典里只有"过去的"元素，绝不会查到自己。
- **Step 5：无解的出口。** 扫完整个数组都没配成对，就返回 `None`。

**手算演示一**（对应 “运行示例”部分 的表格，`a=[2,7,11,15]`、`target=9`）：

| j | x（当前值） | 需寻找（9−x） | 之前字典 | 是否找到 |
|---|---|---|---|---|
| 0 | 2 | 7 | `{}` | 否（存入 2→0） |
| 1 | 7 | 2 | `{2:0}` | 是！返回 `[0,1]` |

第 2 轮就收工，后面的 11、15 根本不用看。

**手算演示二**（重复元素 `[3,3]`、`target=6`，体会"先查后存"的必要性）：

| j | x | 需寻找 | 查询时字典 | 结果 | 存入后字典 |
|---|---|---|---|---|---|
| 0 | 3 | 3 | `{}` | 找不到 | `{3:0}` |
| 1 | 3 | 3 | `{3:0}` | 找到，返回 `[0,1]` | （不再存） |

两个不同位置，答案完全合法。

再对照错误版本（每轮"先存后查"）的执行轨迹：

| j | x | 先存后字典 | 查找 3 | 结果 |
|---|---|---|---|---|
| 0 | 3 | `{3:0}` | 在！ | 返回 `[0,0]` ← 同一元素用了两次，非法 |

看清楚了吗？错误版本在 j=0 这一步就让元素"自己和自己配对"了。"先查后存"四个字的价值就在这里：查询发生时字典里只有过去，当前元素还没资格入账。

## 4. 核心代码：two_sum

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def two_sum(nums, target):
    seen = {}
    for j, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], j]
        seen[x] = j
    return None
```

### 逐段读懂代码

### 函数 1：`two_sum`（哈希版，一遍扫描）

先看扫描与命中返回：

```python
def two_sum(nums, target):
    seen = {}
    for j, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], j]
```

- `seen={}`：准备一个空字典当"账本"，记"见过的值 → 它的下标"。
- `for j,x in enumerate(nums):`：enumerate 同时拿到下标 j 和值 x，一遍从左扫到右。
- `if target-x in seen:`：先算补数，再问字典里有没有这个补数。`in` 对字典做的是按键查找，平均常数时间。注意这行**在存入之前**执行，保证了"只查询过去"。
- `return [seen[target-x],j]`：配对成功。字典里的下标（过去的位置）在前、当前下标 j 在后，天然满足"前面的下标 < 当前的下标"。紧凑写法 `return ...` 直接跟在 if 后面，等价于"条件成立就立刻返回"。

再看记账与无解出口。

```python
seen[x]=j
    return None
```

- `seen[x]=j`：把当前值记进账本。注意如果 x 之前出现过，这一步会用新的下标覆盖旧下标——没关系，题目只要任意一组解，覆盖后存的仍是合法的"过去位置"。
- `return None`：整个数组扫完仍未配对，按本章契约返回 None 表示无解。

### 函数 2：`two_sum_bruteforce`（暴力参照版）

```python
def two_sum_bruteforce(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return None
```

- 外层 `for i` 枚举第一个元素，内层 `for j in range(i+1,...)` 枚举它右边的第二个元素。`i+1` 起步有两重含义：保证 j>i（返回的下标对有序），也保证不把同一个位置和自己配对。
- `if nums[i]+nums[j]==target: return [i,j]`：逐对检查，命中即返回。
- `return None`：与哈希版保持相同的无解契约——这一点很重要，后面的差分测试要拿两者互相对照。
- 这个函数慢，但它"笨得可靠"，正是 N015 教过的参照思路：拿它当标准答案去检验快的版本。

## 5. 分段实现：按顺序运行

**本章接口：** `two_sum(nums, target)`、`two_sum_bruteforce(nums, target)`

### two_sum

In [1]:
def two_sum(nums, target):
    seen = {}
    for j, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], j]
        seen[x] = j
    return None

### two_sum_bruteforce

In [2]:
def two_sum_bruteforce(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return None

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[2,7,11,15]; seen={}; rows=[]
for j,x in enumerate(a):
    rows.append((j,x,9-x,dict(seen),(9-x) in seen)); seen[x]=j
show_table(['j','x','需寻找','之前字典','是否找到'],rows)

j,x,需寻找,之前字典,是否找到
0,2,7,{},False
1,7,2,{2: 0},True
2,11,-2,"{2: 0, 7: 1}",False
3,15,-6,"{2: 0, 7: 1, 11: 2}",False


## 7. 正确性、适用条件与复杂度

考虑任意合法对 i<j：处理 j 时字典中保留了 nums[i] 的某个较早索引，因此一定能找到一个合法对。只查询过去确保索引不同。

**代价：** 哈希版平均 O(n) 时间、O(n) 空间；暴力版 O(n²) 时间、O(1) 空间。

### 展开理解

**为什么是对的：** 关键问题是"如果存在合法解，哈希版一定找得到吗"。假设确实存在一对 i<j 使 `nums[i]+nums[j]=target`。我们让扫描走到 j：无论中途发生了什么，字典里至少会记着 nums[i] 的某个下标（可能是 i 本身，也可能是更早的、值相同的另一个下标——反正是一个合法的过去位置）。到 j 时我们查的是 target−nums[j]，它恰好等于 nums[i]，所以一定查得到，返回的就是一对真实合法的下标。换句话说，扫描只要路过 j，答案就跑不掉。而"两个下标一定不同"由结构保证：查询发生在存入之前、字典里只有左边的元素，所以配对的一方永远来自过去、另一方是当前，位置必然不同。反过来，返回的任何答案都经过了实打实的检查（字典里查到的值加当前 x 正好是 target），所以也不会返回错的东西。

**复杂度是多少：** 哈希版扫一遍数组，每个元素做一次字典查询、一次字典写入，都是平均常数时间，所以平均 O(n) 时间、O(n) 空间（最坏时字典里存了所有 n 个值）。暴力版两层循环要对约 n²/2 对做检查，时间 O(n²)，但只用两个下标变量，空间 O(1)。举个具体的数：n=10⁴ 时暴力要检查约 5000 万对，要等上好几秒；哈希版约两万次字典操作，眨眼完成。这就是"用一个可以忍受的 O(n) 空间，换一个数量级的时间提速"。

**两个高频疑问，顺便说清：**

- 疑问一："先排序再用双指针行不行？"——行，那是另一条 O(n log n) 的经典路线，但它排完序就丢了原始下标，还要多一步记录原位置；哈希版一遍过、下标天然保留，所以在"要返回下标"的契约下更顺手。
- 疑问二："为什么哈希版是'平均'O(n)？"——字典单次查询的常数时间建立在哈希函数把键均匀打散的假设上；极端情况下冲突扎堆，单次查询会退化。日常刷题和面试按平均情形分析就够，但你要知道这个前提存在。

## 8. 单元测试：每个用例在检查什么

```python
assert two_sum([2, 7, 11, 15], 9) == [0, 1]

assert two_sum([3, 3], 6) == [0, 1]

from itertools import product

for n in range(5):
    for a in product([-1, 0, 1], repeat=n):
        for target in range(-2, 3):
            ans = two_sum(a, target)
            ref = two_sum_bruteforce(a, target)
            assert (ans is None) == (ref is None)
            if ans is not None:
                i, j = ans
                assert i < j and a[i] + a[j] == target

print('N019: 所有本章断言通过')
```

- 第 1 条是**正常值测试**（也是 LeetCode 1 的官方样例）：`[2,7,11,15]` 凑 9 应返回 `[0,1]`。这里可以直接断言精确下标，因为这个输入只有唯一解，不存在"多解任取"的歧义。
- 第 2 条是**重复元素测试**（特殊值）：`[3,3]` 凑 6 应返回 `[0,1]`。这条专门钉住"值相同但位置不同、可以配对"的契约，同时检验"先查后存"的顺序没有写反——写反了会返回 `[0,0]`，当场翻车。
- 第三段三重循环是**穷举差分测试**：枚举长度 0~4、元素取 −1/0/1 的全部数组，再配上 target 从 −2 到 2 的每个目标值（组合非常全面：有解、无解、零和、重复值全都被覆盖）。对每种组合让哈希版和暴力参照版各算一遍，断言两件事：
  - 其一，"有没有解"这一判断必须一致：`ans is None` 与 `ref is None` 同真同假，`(ans is None)==(ref is None)` 写的就是这个意思。
  - 其二，如果哈希版给出了答案，答案本身必须合法：`i<j` 保证下标有序且不同，`a[i]+a[j]==target` 保证和真的对。
  - 注意这里**没有**断言两版返回同一对下标——多解输入的契约允许各返回各的合法解，比答案本身更严的比对反而是过度约束。

In [4]:
assert two_sum([2, 7, 11, 15], 9) == [0, 1]

assert two_sum([3, 3], 6) == [0, 1]

from itertools import product

for n in range(5):
    for a in product([-1, 0, 1], repeat=n):
        for target in range(-2, 3):
            ans = two_sum(a, target)
            ref = two_sum_bruteforce(a, target)
            assert (ans is None) == (ref is None)
            if ans is not None:
                i, j = ans
                assert i < j and a[i] + a[j] == target

print('N019: 所有本章断言通过')

N019: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 扩展为返回全部不同索引对。

**练习 2：** 说明不能重复使用同一元素。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（返回全部不同的下标对）**：思路方向——把"找到就 return"改成"找到就收集、继续扫"，最后返回全部合法对。提示：字典里一个值可能对应多个历史下标，所以账本要升级成"值 → 下标列表"；扫描到 x 时把补数对应的每个历史下标都与当前 j 配成一对收进结果。手算例子可用 `[1,1,1]`、target=2，不同下标对共有 (0,1)、(0,2)、(1,2) 三对。别忘了定义输出顺序，方便自测。
- **练习 2（说明不能重复使用同一元素）**：思路方向——这题是让你把"为什么 `two_sum` 不会自己配自己"讲清楚。提示：结合第三节 `[3,3]` 的手算，先演示错误版本（先存后查）在 `[3,3]` 上返回 `[0,0]` 的过程，再解释正确版本"查询发生在存入之前、字典里只有过去"的结构性保证；最后可以用 `[3]`、target=6 这种单元素输入验证：扫完返回 None，3 永远没法和自己凑成 6。

两道练习验收时的共同要求：把"输入、输出、边界"写明白，并且能口头回答"如果输入里同一个值出现三次，账本覆盖会发生什么"——能顺着 `seen[x]=j` 讲清覆盖语义，才算真懂这个字典。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def two_sum(nums, target):
    seen = {}
    for j, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], j]
        seen[x] = j
    return None

def two_sum_bruteforce(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return None

# 示例与回归测试
assert two_sum([2, 7, 11, 15], 9) == [0, 1]

assert two_sum([3, 3], 6) == [0, 1]

from itertools import product

for n in range(5):
    for a in product([-1, 0, 1], repeat=n):
        for target in range(-2, 3):
            ans = two_sum(a, target)
            ref = two_sum_bruteforce(a, target)
            assert (ans is None) == (ref is None)
            if ans is not None:
                i, j = ans
                assert i < j and a[i] + a[j] == target

print('N019: 所有本章断言通过')

N019: 所有本章断言通过


## 11. 代表题与迁移

- **1. Two Sum（两数之和）**：这题就是本章两个函数的原型题——练的是"值到索引的哈希映射 + 补数查询 + 先查后存"这一整套套路，它是后面所有哈希加速思想的第一块基石。
- 迁移建议：这套"边扫边查账本"的骨架能直接搬到 Two Sum 的变体（如 167 两数之和 II、15 三数之和中固定一个数后的两数搜索），也适用于"第一个重复元素""和为 k 的子数组"等一大类题。

**题面入口：** [1](https://leetcode.com/problems/two-sum/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。